# Vote Interpolation

Using the population weights calculated in notebook 03, we allocate historical election results onto the new congressional district boundaries.

The question we are answering: if the 2026 congressional district boundaries had existed during the 2020 presidential election, what would the results have looked like district by district?

The math is simple: for each precinct, multiply the vote totals by the weight for each district fragment, then sum by district.

Sanity check: the total votes for each candidate across all districts must exactly match the original precinct-level totals. If any votes are lost or gained, something went wrong.

## Step 1 — Explore the TLC election data file

Initial attempt using the Texas Legislative Council election analysis file.
Note: this file turned out to contain district-level summaries, not 
precinct-level results. We ended up using the VEST shapefile instead (see Step 4).
Keeping this cell for transparency about the data exploration process.

In [1]:
import pandas as pd

# Load the 2020 election data
df = pd.read_excel('../data/raw/election_results/PLANC2333_r206_Election20G.xls')

print(f"Shape: {df.shape}")
print(df.head(10))

WARNING *** file size (57280) not 512 + multiple of sector size (512)
Shape: (50, 13)
   Unnamed: 0 Unnamed: 1                                         Unnamed: 2  \
0         NaN        NaN                                                NaN   
1         NaN        NaN  Red-206\nData: 2020 Census\nPLANC2333  08/18/2...   
2         NaN        NaN                                                NaN   
3         NaN        NaN                                                NaN   
4         NaN        NaN                                                NaN   
5         NaN        NaN                                                NaN   
6         NaN        NaN                                                NaN   
7         NaN        NaN                                                NaN   
8         NaN        NaN                                                NaN   
9         NaN        NaN                                                NaN   

                 Unnamed: 3 Unnamed: 4  Unna

## Step 2 — Inspect the full TLC file

Prints the entire file to find where actual data starts.
Confirmed: this file only has district-level summaries, not precinct-level results. Not useful for our purposes.

In [2]:
# Print all rows to find where data actually starts
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 60)
print(df.to_string())

    Unnamed: 0 Unnamed: 1                                                     Unnamed: 2                Unnamed: 3 Unnamed: 4  Unnamed: 5 Unnamed: 6                                                  Unnamed: 7 Unnamed: 8  Unnamed: 9  Unnamed: 10  Unnamed: 11                                               Unnamed: 12
0          NaN        NaN                                                            NaN                       NaN        NaN         NaN        NaN                                                         NaN        NaN         NaN          NaN          NaN  Texas Legislative Council\n08/18/25 3:19 PM\nPage 1 of 2
1          NaN        NaN  Red-206\nData: 2020 Census\nPLANC2333  08/18/2025 12:23:12 PM                       NaN        NaN         NaN        NaN                                                         NaN        NaN         NaN          NaN          NaN                                                       NaN
2          NaN        NaN                           

## Step 3 — Explore the Clarity Elections detail file

Second attempt using Travis County's precinct-level export from Clarity Elections.
Note: this file also turned out to be unusable, it had no column headers, making it impossible to identify which column belonged to which candidate. Keeping for transparency.

In [8]:
with zipfile.ZipFile('../data/raw/election_results/detailtxt.zip', 'r') as z:
    print(z.namelist())

['detail.txt']


## Step 4 — Inspect the Clarity Elections file format

Extracts and reads the first 20 lines to understand the structure.
Confirmed: precinct numbers on the left, vote counts across columns, but no headers identifying which column belongs to which candidate. This format is not usable without a separate data dictionary.

In [9]:
with zipfile.ZipFile('../data/raw/election_results/detailtxt.zip', 'r') as z:
    z.extractall('../data/raw/election_results/')

# Read first 20 lines to understand the format
with open('../data/raw/election_results/detail.txt', 'r') as f:
    for i, line in enumerate(f):
        print(line.strip())
        if i > 20:
            break






Registered Voters             Ballots Cast                  Voter Turnout
100                           3375                          1690                          50.07%
101                           3147                          1397                          44.39%
102                           3375                          1804                          53.45%
103                           2343                          1207                          51.52%
104                           3538                          2147                          60.68%
105                           3139                          1602                          51.04%
106                           3348                          2403                          71.77%
107                           3573                          2100                          58.77%
110                           3259                          1489                          45.69%
111                           2847              

## Step 5 — Search for candidate data deeper in the file

Looks further into the file hoping to find labeled candidate columns. Confirmed: still no headers anywhere. 

In [10]:
with open('../data/raw/election_results/detail.txt', 'r') as f:
    lines = f.readlines()

print(f"Total lines: {len(lines)}")

# Look at lines 200-230 to find candidate data
for line in lines[200:230]:
    print(line.strip())

Total lines: 10373
362                           4163                          2565                          61.61%
363                           3069                          1864                          60.74%
364                           2967                          1760                          59.32%
365                           2901                          1865                          64.29%
366                           3705                          2352                          63.48%
367                           2699                          1688                          62.54%
370                           4113                          2243                          54.53%
371                           2802                          1653                          58.99%
372                           795                           447                           56.23%
373                           2280                          1494                          65.53%
374        

## Step 6 — Final check of the Clarity file

Last attempt to find usable structure in the Clarity file.
Confirmed: no column headers anywhere. Moving on to a better data source.

In [11]:
# Look at lines 500-530
for line in lines[500:530]:
    print(line.strip())

243                           3568                          16                            280                           115                           0                             411                           99                            1012                          354                           4                             1469                          0                             35                            19                            0                             54                            0                             1                             1                             0                             2                             1936
244                           2725                          0                             78                            80                            1                             159                           7                             539                           284                           2                             832  

## Step 7 — Check the TLC 2022 election file for precinct data

Third attempt — checking if the TLC 2022 file has a precinct-level sheet. Spoiler: it also only has district-level summaries. But it does contain useful candidate names (Abbott, O'Rourke) that confirm the column structure.

In [14]:
# Check how many sheets are in the TLC 2022 election file
xl = pd.ExcelFile('../data/raw/election_results/PLANC2333_r206_Election22G.xls')
print(xl.sheet_names)

WARNING *** file size (65472) not 512 + multiple of sector size (512)
['Sheet1', 'Sheet2']


## Step 8 — Inspect Sheet2 of the TLC 2022 file

Confirms district-level summaries only. No precinct-level data available from TLC files. Need to find a different source.

In [15]:
df2 = pd.read_excel('../data/raw/election_results/PLANC2333_r206_Election22G.xls', sheet_name='Sheet2')
print(f"Shape: {df2.shape}")
print(df2.head(20))

WARNING *** file size (65472) not 512 + multiple of sector size (512)
Shape: (50, 80)
    Unnamed: 0        Unnamed: 1  \
0          NaN               NaN   
1          NaN               NaN   
2          NaN               NaN   
3          NaN               NaN   
4          NaN               NaN   
5          NaN               NaN   
6          NaN               NaN   
7          NaN               NaN   
8          NaN          District   
9          NaN               NaN   
10         NaN               NaN   
11         NaN             STATE   
12         NaN                 1   
13         NaN                 2   
14         NaN                 3   
15         NaN                 4   
16         NaN                 5   
17         NaN                 6   
18         NaN                 7   
19         NaN                 8   

                                           Unnamed: 2  Unnamed: 3  Unnamed: 4  \
0                                                 NaN         NaN         Na

## Step 9 — Inspect the VEST shapefile

Fourth attempt — using the Voting and Election Science Team (VEST) shapefile from Harvard Dataverse. This turned out to be the right source. VEST provides clean, standardized precinct-level election results as a shapefile, meaning it has both vote counts and precinct boundaries in one file.

In [ ]:
#Another Try

In [16]:
with zipfile.ZipFile('../data/raw/election_results/tx_2020.zip', 'r') as z:
    print(z.namelist())

['tx_2020.cpg', 'tx_2020.dbf', 'tx_2020.prj', 'tx_2020.shp', 'tx_2020.shx']


## Step 10 — Load the VEST shapefile

Loads the statewide Texas 2020 election results. Key columns:
- PCTKEY: precinct ID (matches our boundary files)
- G20PREDBID: Biden votes
- G20PRERTRU: Trump votes
- G20PRELJOR: Jo Jorgensen (Libertarian) votes
- G20PREGHAW: Howie Hawkins (Green) votes

Same CRS as all our other files, no reprojection needed.

In [17]:
import geopandas as gpd

vest = gpd.read_file('zip://../data/raw/election_results/tx_2020.zip!tx_2020.shp')

print(f"Shape: {vest.shape}")
print(f"CRS: {vest.crs}")
print(f"Columns: {list(vest.columns)}")
print(vest.head(3))

Shape: (9016, 38)
CRS: PROJCS["NAD_1983_Lambert_Conformal_Conic",GEOGCS["NAD83",DATUM["North_American_Datum_1983",SPHEROID["GRS 1980",6378137,298.257222101,AUTHORITY["EPSG","7019"]],AUTHORITY["EPSG","6269"]],PRIMEM["Greenwich",0],UNIT["Degree",0.0174532925199433],AUTHORITY["EPSG","4269"]],PROJECTION["Lambert_Conformal_Conic_2SP"],PARAMETER["latitude_of_origin",31.1666666666667],PARAMETER["central_meridian",-100],PARAMETER["standard_parallel_1",27.4166666666667],PARAMETER["standard_parallel_2",34.9166666666667],PARAMETER["false_easting",1000000],PARAMETER["false_northing",1000000],UNIT["metre",1,AUTHORITY["EPSG","9001"]],AXIS["Easting",EAST],AXIS["Northing",NORTH]]
Columns: ['CNTY', 'COLOR', 'PREC', 'PCTKEY', 'CNTYKEY', 'G20VR', 'G20SSVR', 'G20PRERTRU', 'G20PREDBID', 'G20PRELJOR', 'G20PREGHAW', 'G20PREOWRI', 'G20USSRCOR', 'G20USSDHEG', 'G20USSLMCK', 'G20USSGCOL', 'G20RRCRWRI', 'G20RRCDCAS', 'G20RRCLSTE', 'G20RRCGGRU', 'G20SSCRHEC', 'G20SSCDMEA', 'G20SSCLASH', 'G20SSCRBLA', 'G20SSCDCHE',

## Step 11 — Filter VEST data to Travis County and validate

Filters to Travis County's 247 precincts and prints total vote counts. Quick gut check: Travis County leaned heavily Democratic in 2020 (Biden 435,860 vs Trump 161,337 — about 73% Democratic).
If these numbers look wrong, something went wrong in the filter.

In [18]:
# Filter to Travis County
travis_vest = vest[vest['CNTY'] == 453].copy()

print(f"Travis County precincts: {len(travis_vest)}")
print(f"\nTotal Biden votes: {travis_vest['G20PREDBID'].sum():,.0f}")
print(f"Total Trump votes: {travis_vest['G20PRERTRU'].sum():,.0f}")
print(f"\nSample:")
travis_vest[['PCTKEY', 'G20PREDBID', 'G20PRERTRU', 'G20PRELJOR']].head(5)

Travis County precincts: 247

Total Biden votes: 435,860
Total Trump votes: 161,337

Sample:


,PCTKEY,G20PREDBID,G20PRERTRU,G20PRELJOR
809,4530128,0,0,0
810,4530131,1,0,0
811,4530134,0,0,0
812,4530118,377,47,2
813,4530123,4695,2361,133


## Step 12 — Load weights and reshape vote data

Loads the population weights from notebook 03, then reshapes the vote data from wide format (one row per precinct, one column per candidate) to long format (one row per precinct per candidate).

Wide format example:
PCTKEY | Biden | Trump
4530101 | 1240 | 880

Long format example:
PCTKEY | candidate | votes
4530101 | Biden | 1240
4530101 | Trump | 880

Long format makes the join and calculation much simpler.

In [19]:
# Load population weights
weights = pd.read_csv('../data/processed/travis_population_weights.csv')

# Reshape vote data to long format (one row per precinct per candidate)
travis_votes = travis_vest[['PCTKEY', 'G20PREDBID', 'G20PRERTRU', 'G20PRELJOR', 'G20PREGHAW']].copy()
travis_votes.columns = ['PCTKEY', 'Biden', 'Trump', 'Jorgensen', 'Hawkins']

# Melt to long format
votes_long = travis_votes.melt(id_vars='PCTKEY', var_name='candidate', value_name='votes')

print(f"Vote rows: {len(votes_long)}")
print(votes_long.head(8))

Vote rows: 988
    PCTKEY candidate  votes
0  4530128     Biden      0
1  4530131     Biden      1
2  4530134     Biden      0
3  4530118     Biden    377
4  4530123     Biden   4695
5  4530222     Biden   1690
6  4530227     Biden    554
7  4530267     Biden    928


## Step 13 — Join votes to weights and calculate allocated votes

Joins the vote data to the population weights on precinct ID, then applies the core calculation:

allocated_votes = votes x weight

For example: if precinct 4530101 gave Biden 1,240 votes and 99.3% of that precinct falls in District 37, then District 37 gets 1,240 x 0.993 = 1,231 estimated Biden votes from that precinct.

Finally groups by district and candidate to get the total estimated votes per district. Results show all 7 districts across 4 candidates = 28 rows.

In [21]:
# Join votes to weights on precinct ID
votes_long['PCTKEY'] = votes_long['PCTKEY'].astype(str)
weights['old_precinct_id'] = weights['old_precinct_id'].astype(str)

merged = votes_long.merge(weights, left_on='PCTKEY', right_on='old_precinct_id', how='left')

print(f"Merged rows: {len(merged)}")
print(f"Null weights: {merged['weight'].isna().sum()}")

# Calculate allocated votes
merged['allocated_votes'] = merged['votes'] * merged['weight']

# Group by district and candidate
results = merged.groupby(['new_district_id', 'candidate'])['allocated_votes'].sum().reset_index()
results.columns = ['new_district_id', 'candidate', 'estimated_votes']
results['estimated_votes'] = results['estimated_votes'].round(1)

print(f"\nResults shape: {results.shape}")
print(results.sort_values(['new_district_id', 'candidate']))

Merged rows: 1672
Null weights: 0

Results shape: (28, 3)
    new_district_id  candidate  estimated_votes
0                10      Biden          86071.1
1                10    Hawkins            334.5
2                10  Jorgensen           2122.5
3                10      Trump          60596.2
4                11      Biden          45257.7
5                11    Hawkins            213.3
6                11  Jorgensen           1174.9
7                11      Trump          24427.4
8                17      Biden           1721.2
9                17    Hawkins              7.5
10               17  Jorgensen             49.7
11               17      Trump           1376.2
12               21      Biden            366.6
13               21    Hawkins              1.3
14               21  Jorgensen             10.6
15               21      Trump            315.6
16               27      Biden          25147.7
17               27    Hawkins            129.2
18               27  Jorgensen

In [22]:
# Sanity check: total votes must match original
original_biden = travis_vest['G20PREDBID'].sum()
original_trump = travis_vest['G20PRERTRU'].sum()

estimated_biden = results[results['candidate'] == 'Biden']['estimated_votes'].sum()
estimated_trump = results[results['candidate'] == 'Trump']['estimated_votes'].sum()

print(f"Original Biden:   {original_biden:,.0f}")
print(f"Estimated Biden:  {estimated_biden:,.0f}")
print(f"Difference:       {abs(original_biden - estimated_biden):.1f}")
print()
print(f"Original Trump:   {original_trump:,.0f}")
print(f"Estimated Trump:  {estimated_trump:,.0f}")
print(f"Difference:       {abs(original_trump - estimated_trump):.1f}")

Original Biden:   435,860
Estimated Biden:  435,860
Difference:       0.0

Original Trump:   161,337
Estimated Trump:  161,337
Difference:       0.0


In [23]:
results.to_csv('../data/processed/travis_interpolated_results.csv', index=False)
print("Saved to /data/processed/travis_interpolated_results.csv")

Saved to /data/processed/travis_interpolated_results.csv
